# Phase 02 — Step 14
## Live Evidence-Grounded OpenAI Recommendation System

Notebook 10 already designed the Phase 02 recommendation system:

1. frozen Linear Regression model predicts employability
2. construct scores, item scores, and SHAP become evidence
3. OpenAI converts that evidence into guidance
4. a rule-based fallback is used if the API fails

This notebook documents the live service integration that applies the same design to the improved frozen Phase 02 model (OOF R² 0.926, 1,500 rows).

The language model does **not** calculate or change `EVS_avg`. The OpenAI key is loaded only from `samindi/Employability/.env`.

Notebooks 01–12 remain the original 1,000-row development evidence. Phase 01 and Phase 03 services are not modified here.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "phase2_competence":
    PROJECT_DIR = PROJECT_DIR.parent.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from employability_service import build_phase_services

services = build_phase_services(PROJECT_DIR)
phase2 = services["phase2"]
print("Phase:", phase2.phase_label)
print("Model loaded:", phase2.exists())
print("Model type:", phase2.model_type)

## Sample competence profile

This mixed profile is the same style as notebook 10: stronger IT & Media items, developing Learning & Innovation items.

In [ ]:
sample_responses = {
    "C1": 4, "C2": 4, "C3": 3, "C4": 4,
    "C5": 3, "C6": 4, "C7": 3,
    "C8": 5, "C9": 4, "C10": 4,
}

result = phase2.predict(sample_responses)

print("Predicted EVS_avg:", result["prediction"]["evs_avg"])
print("Recommendation source:", result.get("recommendation_source"))
print("Construct scores:", result["construct_scores"])
print("SHAP:", result.get("shap_explanation"))

In [ ]:
guidance = result.get("ai_recommendation", {})

print(guidance.get("overall_career_readiness_summary", ""))
print("\nTop strengths:")
for item in guidance.get("top_3_strengths", []):
    print(" -", item)
print("\nPriority gaps:")
for item in guidance.get("top_3_areas_for_improvement", []):
    print(" -", item)

## Evidence contract

- Score source: frozen Linear Regression + StandardScaler
- Features: `LifeCareer_avg`, `LearningInnovation_avg`, `ITMedia_avg`
- Explanation: Linear SHAP on the scaled constructs
- Guidance: OpenAI JSON schema from notebook 10, with rule-based fallback
- Live module: `phase2_openai_guidance.py`